In [ ]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.kafka_consumer_adapter import run_kafka_consumer_to_postgres_once, run_kafka_consumer_to_postgres_loop

In [ ]:
engine = get_engine_from_env()

### Single Batch

In [ ]:

result = run_kafka_consumer_to_postgres_once(
    engine=engine,
    topic="pump.telemetry.synthetic",
    schema="capstone",
    table_name="synthetic_sensor_messages_consumed_stage",
    max_messages=500,
    poll_timeout_seconds=1.0,
    consumer_group_id="synthetic-telemetry-consumer-group",
    consumer_worker_id="consumer_worker_001",
    auto_offset_reset="earliest",
)


In [ ]:

result

----

### Loop Batches

In [ ]:
results = run_kafka_consumer_to_postgres_loop(
    engine=engine,
    topic="pump.telemetry.synthetic",
    schema="capstone",
    table_name="synthetic_sensor_messages_consumed_stage",
    max_messages=500,
    poll_timeout_seconds=1.0,
    consumer_group_id="synthetic-telemetry-consumer-group",
    consumer_worker_id="consumer_worker_001",
    auto_offset_reset="earliest",
    max_batches=10,
    idle_sleep_seconds=0.0,
    stop_on_empty=True,
)


In [ ]:

results

----

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT observation_index) AS distinct_observation_count,
        COUNT(DISTINCT kafka_topic || ':' || kafka_partition || ':' || kafka_offset) AS distinct_kafka_messages,
        MIN(consumer_received_at) AS min_consumer_received_at,
        MAX(consumer_received_at) AS max_consumer_received_at
    FROM capstone.synthetic_sensor_messages_consumed_stage
    """
)


In [ ]:

display(validation_dataframe)